# RAG and its types.

since you are already working with Ollama + Jupyter + local embeddings, I’ll show you the practical RAG variants that actually matter for you and give you ready-to-run code for each using:

LLM → gemma3:1b or llama3.2:1b

Embeddings → nomic-embed-text:latest

Backend → Ollama

Framework → modern LangChain v0.2+

| RAG type         | Why it exists                    |
| ---------------- | -------------------------------- |
| Basic RAG        | Baseline – works for most cases  |
| Multi-query RAG  | Fixes bad user questions         |
| Parent–Child RAG | Fixes lost context in chunking   |
| Hybrid RAG       | Combines keyword + vector search |
| Reranking RAG    | Improves retrieval quality       |


In [ ]:
# pip install -U langchain langchain-community langchain-text-splitters chromadb

# Importing and Ollama models

In [1]:
from langchain_community.llms import Ollama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

c:\Users\saksh\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### LLM and Embeddings

In [2]:
llm = Ollama(model="smollm2:360m")   # or llama3.2:1b

embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest"
)

C:\Users\saksh\AppData\Local\Temp\ipykernel_17448\1112943999.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="smollm2:360m")   # or llama3.2:1b
C:\Users\saksh\AppData\Local\Temp\ipykernel_17448\1112943999.py:3: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(


### Loading your documents.

In [4]:
loader = TextLoader("data.txt", encoding="utf-8")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

splits = splitter.split_documents(docs)

Basic RAG (standard RAG)

Architecture

Query → Vector DB → Top-k chunks → LLM

#### Build vector store

In [5]:
db = Chroma.from_documents(
    splits,
    embedding=embeddings
)

retriever = db.as_retriever(search_kwargs={"k": 4})

#### RAG Chain

In [ ]:
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context.

Context:
{context}

Question:
{question}
""")

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": lambda x: x
    }
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("Tell me about blogging?")

'Blogging refers to the practice of writing articles or posts for an online audience on various topics related to a specific niche, hobby, or profession. This platform allows individuals to share their thoughts, experiences, and expertise with like-minded readers, fostering connections between people who are passionate about similar subjects. Blogging can be done through written content such as text posts, videos, podcasts, or even interactive experiences like quizzes and games. By publishing regularly, blogs create a community around the topic, encourage engagement from their followers, and help establish them as authority figures in their fields.'